In [2]:
import pandas as pd
import numpy as np
import subprocess
import sys
import mlflow

In [3]:
# Carregando o dataset tratado para o repositório
df = pd.read_parquet('../data/processed/dataset_tratado.parquet')

In [ ]:
dataset = mlflow.data.dataset(df,name="Churn_Full_Dataset")

In [4]:
import mlflow

# Define o banco de dados na raiz do projeto
mlflow.set_tracking_uri("sqlite:///../mlflow.db")

# Cria ou seleciona um experimento com nome específico
mlflow.set_experiment("Projeto_Churn_Tech_Challenge")


<Experiment: artifact_location=('file:c:/Users/lara-/OneDrive/Desktop/POS TECH '
 'ML/projeto-tech-challenge-churn/notebooks/mlruns/1'), creation_time=1774800663369, experiment_id='1', last_update_time=1774800663369, lifecycle_stage='active', name='Projeto_Churn_Tech_Challenge', tags={}, workspace='default'>

In [5]:
mlflow.get_tracking_uri()

'sqlite:///../mlflow.db'

# --------------- REGRESSÃO LOGISTICA ---------------

In [6]:
import pandas as pd
import mlflow
import mlflow.sklearn
import mlflow.data
import matplotlib.pyplot as plt # Adicionado para o gráfico
import os # Adicionado para gerenciar arquivos
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, precision_score, 
                             recall_score, roc_auc_score, confusion_matrix, 
                             classification_report, ConfusionMatrixDisplay) # Adicionado ConfusionMatrixDisplay

# 1. Configurações de separação
test_size = 0.2
random_state = 42

# Separando variáveis alvo e features
X = df.drop(columns=['churn_value'])
y = df['churn_value']

# Split com estratificação (importante para Churn!)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=test_size,
    random_state=random_state,
    stratify=y
)

# 2. Criando o objeto de dataset do MLflow (para a aba "Inputs")
# Certifique-se de que a variável 'df' ainda contenha o dataframe original
# Use o caminho absoluto ou relativo correto para o seu CSV
csv_path = "data/churn_data.csv"
dataset = mlflow.data.from_pandas(df, source=csv_path, name="Churn_Analysis_Base")

# Preprocessamento
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X_train.select_dtypes(include='object').columns

preprocessador = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
        ('num', StandardScaler(), numerical_cols)
    ]
)

# Aplicando transformação
X_train_transformado = preprocessador.fit_transform(X_train)
X_test_transformado = preprocessador.transform(X_test)

feature_names = preprocessador.get_feature_names_out()
X_train_df = pd.DataFrame(X_train_transformado, columns=feature_names)
X_test_df = pd.DataFrame(X_test_transformado, columns=feature_names)

# --- INÍCIO DO EXPERIMENTO NO MLFLOW ---
# Nome amigável para a sua Run
run_name_input = "regressao_logistica_baseline"

with mlflow.start_run(run_name=run_name_input) as run:
    
    # Definindo o modelo
    model = LogisticRegression(random_state=random_state, max_iter=1000)
    
    # 1. Logando o Dataset (Aba Inputs)
    mlflow.log_input(dataset, context="training")
    
    # 2. Logando Parâmetros (Aba Parameters)
    mlflow.log_params({
        "model_type": "LogisticRegression",
        "max_iter": 1000,
        "random_state": random_state,
        "test_size": test_size,
        "stratify": "True",
        "train_samples": len(X_train),
        "test_samples": len(X_test)
    })

    # 3. Treinando o modelo
    model.fit(X_train_df, y_train)

    # 4. Logando o modelo treinado (Aba Artifacts)
    # É melhor logar DEPOIS do fit para salvar os pesos
    mlflow.sklearn.log_model(model, "logistic_regression_model")

    # Predições e métricas
    y_pred_test = model.predict(X_test_df)
    y_probs = model.predict_proba(X_test_df)[:, 1]

    # Matriz de Confusão numérica
    cm = confusion_matrix(y_test, y_pred_test)

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred_test),
        "precision": precision_score(y_test, y_pred_test),
        "recall": recall_score(y_test, y_pred_test),
        "f1_score": f1_score(y_test, y_pred_test),
        "roc_auc": roc_auc_score(y_test, y_probs)
    }

    # 5. Logando Métricas (Aba Metrics)
    mlflow.log_metrics(metrics)

    # 6. Gerando e Logando o Gráfico da Matriz de Confusão (Aba Artifacts)
    # Define o nome do arquivo temporário
    plot_file_name = "confusion_matrix.png"
    
    # Gera o gráfico
    fig, ax = plt.subplots(figsize=(8, 6))
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn', 'Churn']).plot(ax=ax, cmap='Blues')
    ax.set_title(f'Matriz de Confusão - {run_name_input}')
    
    # Salva o gráfico localmente
    plt.savefig(plot_file_name)
    plt.close(fig) # Fecha o plot para liberar memória
    
    # Salva o gráfico como artefato no MLflow
    mlflow.log_artifact(plot_file_name)
    
    # (Opcional) Remove o arquivo temporário local
    if os.path.exists(plot_file_name):
        os.remove(plot_file_name)

    # Output no console
    print(f"Logistic Regression - Métricas de validação")
    print("-" * 30)
    for k, v in metrics.items():
        print(f"{k.capitalize()}: {v:.4f}")
    print("-" * 30)
    print("Confusion Matrix (Numeric):\n", cm)
    print("Classification Report:\n", classification_report(y_test, y_pred_test))

c:\Users\lara-\anaconda3\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
c:\Users\lara-\anaconda3\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest

Logistic Regression - Métricas de validação
------------------------------
Accuracy: 0.9681
Precision: 0.9970
Recall: 0.8824
F1_score: 0.9362
Roc_auc: 0.9798
------------------------------
Confusion Matrix (Numeric):
 [[1034    1]
 [  44  330]]
Classification Report:
               precision    recall  f1-score   support

           0       0.96      1.00      0.98      1035
           1       1.00      0.88      0.94       374

    accuracy                           0.97      1409
   macro avg       0.98      0.94      0.96      1409
weighted avg       0.97      0.97      0.97      1409



In [ ]:
# A avaliação da regressão logística já está feita no bloco de treino principal
# Este bloco é mantido para compatibilidade, mas não reexecuta o mesmo run.

print("Logistic Regression model already trained and métricas logadas no MLflow no bloco anterior.")


# --------------- DUMMY CLASSIFIER ---------------

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline


# =========================
# Separando features e alvo
# =========================
X = df.drop(columns=['churn_value'])
y = df['churn_value']

# =========================
# Split treino e teste
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =========================
# Identificando colunas
# =========================
numerical_cols = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X_train.select_dtypes(include='object').columns

# =========================
# One Hot Encoder
# =========================
encoder = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

# =========================
# Preprocessador
# =========================
preprocessador = ColumnTransformer(
    transformers=[
        ('cat', encoder, categorical_cols),
        ('num', StandardScaler(), numerical_cols)
    ]
)

# =========================
# Seed
# =========================
SEED = 20
np.random.seed(SEED)

# =========================
# Dummy Classifier
# =========================
modelo_dummy = DummyClassifier(
    strategy='stratified',
    random_state=SEED
)


with mlflow.start_run(run_name="Dummy_Classifier_baseline"):

    mlflow.log_params({
        "model_type": "DummyClassifier",
        "max_iter": 1000,
        "random_state": 42
    })

    mlflow.sklearn.log_model(modelo_dummy, "dummy_classifier_model")

# =========================
# Pipeline (boa prática)
# =========================
pipeline_dummy = Pipeline([
    ('preprocessamento', preprocessador),
    ('modelo', modelo_dummy)
])

# =========================
# Validação cruzada
# =========================
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=SEED
)

resultados = cross_validate(
    pipeline_dummy,
    X_train,
    y_train,
    cv=cv,
    scoring=['accuracy', 'precision', 'recall', 'f1'],
    return_train_score=True
)

# =========================
# Resultados
# =========================
print("===== Dummy Classifier =====")
print("Accuracy média:", resultados['test_accuracy'].mean())
print("F1 média:", resultados['test_f1'].mean())
print("Precision média:", resultados['test_precision'].mean())
print("Recall média:", resultados['test_recall'].mean())

mlflow.log_metrics({
    "dummy_accuracy": resultados['test_accuracy'].mean(),
    "dummy_precision": resultados['test_precision'].mean(),
    "dummy_recall": resultados['test_recall'].mean(),
    "dummy_f1": resultados['test_f1'].mean()})          

# --------------- REGRESSÃO LINEAR ---------------

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np

with mlflow.start_run(run_name="linear_regression"):
    lr_model = LinearRegression()
    mlflow.log_param("model_type", "LinearRegression")

    lr_model.fit(X_train_transformado, y_train)

    y_pred_train_lr = lr_model.predict(X_train_transformado)
    y_pred_test_lr = lr_model.predict(X_test_transformado)

    mse_train = mean_squared_error(y_train, y_pred_train_lr)
    mse_test = mean_squared_error(y_test, y_pred_test_lr)
    rmse_train = np.sqrt(mse_train)
    rmse_test = np.sqrt(mse_test)
    mae_train = mean_absolute_error(y_train, y_pred_train_lr)
    mae_test = mean_absolute_error(y_test, y_pred_test_lr)
    r2_train = r2_score(y_train, y_pred_train_lr)
    r2_test = r2_score(y_test, y_pred_test_lr)

    mlflow.log_metrics({
        "mse_train": mse_train,
        "mse_test": mse_test,
        "rmse_train": rmse_train,
        "rmse_test": rmse_test,
        "mae_train": mae_train,
        "mae_test": mae_test,
        "r2_train": r2_train,
        "r2_test": r2_test
    })

    mlflow.sklearn.log_model(lr_model, "linear_regression_model")

    print("Linear Regression - Métricas")
    print(f"RMSE Treino: {rmse_train:.4f}")
    print(f"RMSE Teste: {rmse_test:.4f}")
    print(f"MSE Treino: {mse_train:.4f}")
    print(f"MSE Teste: {mse_test:.4f}")
    print(f"MAE Treino: {mae_train:.4f}")
    print(f"MAE Teste: {mae_test:.4f}")
    print(f"R² Treino: {r2_train:.4f}")
    print(f"R² Teste: {r2_test:.4f}")

# --------------- ÁRVORE DE DECISÃO ---------------

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

with mlflow.start_run(run_name="decision_tree"):
    modelo_tree = DecisionTreeClassifier(max_depth=5, random_state=42)
    mlflow.log_params({
        "model_type": "DecisionTreeClassifier",
        "max_depth": 5,
        "random_state": 42
    })

    modelo_tree.fit(X_train_transformado, y_train)

    y_pred_train_tree = modelo_tree.predict(X_train_transformado)
    y_pred_test_tree = modelo_tree.predict(X_test_transformado)

    acc_train_tree = accuracy_score(y_train, y_pred_train_tree)
    acc_test_tree = accuracy_score(y_test, y_pred_test_tree)
    prec_tree = precision_score(y_test, y_pred_test_tree)
    rec_tree = recall_score(y_test, y_pred_test_tree)
    f1_tree = f1_score(y_test, y_pred_test_tree)
    roc_auc_tree = roc_auc_score(y_test, modelo_tree.predict_proba(X_test_transformado)[:, 1])
    cm_tree = confusion_matrix(y_test, y_pred_test_tree)
    overfitting_tree = acc_train_tree - acc_test_tree

    mlflow.log_metrics({
        "accuracy_train": acc_train_tree,
        "accuracy_test": acc_test_tree,
        "precision": prec_tree,
        "recall": rec_tree,
        "f1_score": f1_tree,
        "roc_auc": roc_auc_tree,
        "overfitting": overfitting_tree
    })

    mlflow.sklearn.log_model(modelo_tree, "decision_tree_model")

    print("Decision Tree - Métricas")
    print(f"Accuracy Treino: {acc_train_tree:.4f}")
    print(f"Accuracy Teste: {acc_test_tree:.4f}")
    print(f"Precision: {prec_tree:.4f}")
    print(f"Recall: {rec_tree:.4f}")
    print(f"F1-score: {f1_tree:.4f}")
    print(f"ROC AUC: {roc_auc_tree:.4f}")
    print(f"Overfitting: {overfitting_tree:.4f}")
    print("Confusion Matrix:\n", cm_tree)
    print("Classification Report:\n", classification_report(y_test, y_pred_test_tree))


# --------------- RANDOM FOREST ---------------

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix, classification_report

with mlflow.start_run(run_name="random_forest"):
    rf_model = RandomForestClassifier(random_state=42, n_estimators=100)
    mlflow.log_params({
        "model_type": "RandomForestClassifier",
        "n_estimators": 100,
        "random_state": 42
    })

    rf_model.fit(X_train_transformado, y_train)

    y_pred_train_rf = rf_model.predict(X_train_transformado)
    y_pred_test_rf = rf_model.predict(X_test_transformado)

    acc_train_rf = accuracy_score(y_train, y_pred_train_rf)
    acc_test_rf = accuracy_score(y_test, y_pred_test_rf)
    prec_rf = precision_score(y_test, y_pred_test_rf)
    rec_rf = recall_score(y_test, y_pred_test_rf)
    f1_rf = f1_score(y_test, y_pred_test_rf)
    roc_auc_rf = roc_auc_score(y_test, rf_model.predict_proba(X_test_transformado)[:, 1])
    cm_rf = confusion_matrix(y_test, y_pred_test_rf)
    overfitting_rf = acc_train_rf - acc_test_rf

    mlflow.log_metrics({
        "accuracy_train": acc_train_rf,
        "accuracy_test": acc_test_rf,
        "precision": prec_rf,
        "recall": rec_rf,
        "f1_score": f1_rf,
        "roc_auc": roc_auc_rf,
        "overfitting": overfitting_rf
    })

    mlflow.sklearn.log_model(rf_model, "random_forest_model")

    print("Random Forest - Métricas")
    print(f"Accuracy Treino: {acc_train_rf:.4f}")
    print(f"Accuracy Teste: {acc_test_rf:.4f}")
    print(f"Precision: {prec_rf:.4f}")
    print(f"Recall: {rec_rf:.4f}")
    print(f"F1-score: {f1_rf:.4f}")
    print(f"ROC AUC: {roc_auc_rf:.4f}")
    print(f"Overfitting: {overfitting_rf:.4f}")
    print("Confusion Matrix:\n", cm_rf)
    print("Classification Report:\n", classification_report(y_test, y_pred_test_rf))


In [ ]:
# Resumo de todos os experimentos MLflow
experiment = mlflow.get_experiment_by_name('churn_prediction_models')

if experiment is None:
    print("Experimento 'churn_prediction_models' não encontrado.")
    print("Verifique se o experimento foi criado antes de buscar runs.")
else:
    runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
    print("RESUMO DE TODOS OS MODELOS TREINADOS")
    print("="*80)

    if runs.empty:
        print("Nenhuma run encontrada para o experimento.")
    else:
        for idx, run in runs.iterrows():
            print(f"\nRun: {run['tags.mlflow.runName']}")
            print(f"Status: {run['status']}")
            print("Métricas:")
            for col in runs.columns:
                if col.startswith('metrics.'):
                    metric_name = col.replace('metrics.', '')
                    metric_value = run[col]
                    if pd.notna(metric_value):
                        print(f"  {metric_name}: {metric_value:.4f}")
